# Global LightGBM Forecasting
[Open in Colab](https://colab.research.google.com/github/ericmavigo/retail-demand-forecasting/blob/main/notebooks/03_lightgbm_forecasting.ipynb)

Train a reproducible global model, preserve the final 28 days as an unseen holdout, and compare transparent hybrids with the statistical baseline.

In [ ]:
# %pip install -r ../requirements-dev.txt
from pathlib import Path
import subprocess,sys,pandas as pd
import plotly.express as px
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()

## Experiment configuration
The random seed and 800,000 sampled training examples make the experiment repeatable. Features include lags, rolling demand, calendar signals and product/store identifiers.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'src/train_model.py'),'--data-dir',str(ROOT/'data/raw'),'--output-dir',str(ROOT/'data/processed'),'--reports-dir',str(ROOT/'reports'),'--samples','800000'],check=True)

In [ ]:
metrics=pd.read_csv(ROOT/'reports/model_metrics.csv').sort_values('WAPE')
metrics.style.format({'MAE':'{:.4f}','WAPE':'{:.2%}','RMSSE':'{:.4f}','Bias':'{:.2%}'})

In [ ]:
plot=metrics.assign(WAPE_percent=100*metrics.WAPE)
px.bar(plot.sort_values('WAPE_percent',ascending=False),x='WAPE_percent',y='model',orientation='h',text_auto='.2f',title='28-day holdout WAPE').show()

In [ ]:
fc=pd.read_csv(ROOT/'data/processed/forecast_daily.csv',parse_dates=['date']).melt('date',var_name='series',value_name='units')
px.line(fc,x='date',y='units',color='series',title='Actual versus forecast demand').show()
pd.read_csv(ROOT/'data/processed/feature_importance.csv').head(15)

## Interpretation
The pure LightGBM model does not beat the moving-average baseline. A 25% LightGBM hybrid improves WAPE and RMSSE slightly. Keeping the weaker model visible demonstrates honest evaluation rather than cherry-picking.